# Dictionnaire CNN — Théorie & Code Python
## Projet DS Leyenda — TouNum

---

Ce notebook est un **dictionnaire de référence** pour les réseaux de neurones convolutifs (CNNs).
Chaque concept est défini théoriquement, puis illustré par le code Python correspondant.

### Structure

| Partie | Contenu |
|--------|---------|
| **1. Données & images** | Pixel, tenseur, normalisation, batch |
| **2. Couches CNN** | Conv2D, Pooling, BN, Dropout, Dense… |
| **3. Fonctions d'activation** | ReLU, Softmax, Sigmoid |
| **4. Apprentissage** | Loss, optimizer, LR, epoch, overfitting… |
| **5. Techniques de régularisation** | Augmentation, class weights, early stopping… |
| **6. Pipeline tf.data** | Dataset, map, prefetch, shuffle… |
| **7. API Keras** | model.compile, fit, evaluate, predict… |
| **8. Évaluation** | Accuracy, matrice de confusion, F1-score… |
| **9. Transfer Learning** | Feature extraction, fine-tuning, architectures pré-entraînées |
| **10. Autoencodeurs & VAE** | Encoder, décodeur, espace latent, génération |
| **11. RNN & Transformers** | LSTM, GRU, attention, BERT, ViT |
| **12. Captioning d'image** | Show&Tell, CLIP, BLIP, BLEU, CIDEr |

---
# Partie 1 — Données & Images

## Pixel

Un **pixel** (picture element) est la plus petite unité d'une image numérique.  
Sa valeur représente une intensité lumineuse :
- **Image en niveaux de gris** : 1 valeur par pixel, entre 0 (noir) et 255 (blanc)
- **Image couleur (RGB)** : 3 valeurs par pixel — Rouge, Vert, Bleu — chacune entre 0 et 255

Une image de 128×128 pixels en RGB contient donc **128 × 128 × 3 = 49 152 valeurs**.

## Tenseur

Un **tenseur** est une généralisation des matrices à n dimensions. En deep learning :

| Dimensions | Nom courant | Exemple |
|------------|-------------|----------|
| 1D | Vecteur | `[0.2, 0.5, 0.3]` (probabilités) |
| 2D | Matrice | Image en niveaux de gris (H×W) |
| 3D | Volume | Image couleur (H×W×C) |
| 4D | Batch d'images | (N×H×W×C) — N images simultanées |

TensorFlow et NumPy manipulent des tenseurs nativement.

```
Image 128×128 RGB  →  shape (128, 128, 3)
Batch de 32 images →  shape (32, 128, 128, 3)
```

In [ ]:
import numpy as np
import tensorflow as tf

# Créer un tenseur représentant une image couleur 128x128
image = np.random.randint(0, 256, size=(128, 128, 3), dtype=np.uint8)
print(f"Shape de l'image    : {image.shape}")   # (128, 128, 3)
print(f"Type des valeurs    : {image.dtype}")
print(f"Valeur min / max    : {image.min()} / {image.max()}")

# Ajouter la dimension batch (nécessaire pour passer dans un modèle)
batch = np.expand_dims(image, axis=0)          # shape (1, 128, 128, 3)
print(f"\nShape avec batch    : {batch.shape}")  # (1, 128, 128, 3)

## Normalisation

Les pixels ont des valeurs brutes dans **[0, 255]**. Les réseaux de neurones convergent mieux quand les entrées sont dans **[0, 1]** ou **[-1, 1]**, car :
- Les gradients ont des magnitudes comparables entre les features
- La descente de gradient est plus stable

**Normalisation simple (MinMax) :** diviser par 255
$$x_{norm} = \frac{x}{255}$$

**Standardisation :** soustraire la moyenne, diviser par l'écart-type (utilisé en transfer learning)

In [ ]:
from tensorflow.keras import layers

# ---- Option 1 : couche Rescaling dans le modèle ----
rescale = layers.Rescaling(1.0 / 255.0)
# Divise chaque pixel par 255 → valeurs dans [0, 1]

# ---- Option 2 : en numpy (hors modèle) ----
image_float = image.astype(np.float32) / 255.0
print(f"Avant normalisation : min={image.min()}, max={image.max()}")
print(f"Après normalisation : min={image_float.min():.3f}, max={image_float.max():.3f}")

## Batch (mini-batch)

Au lieu de présenter les images **une par une** (stochastique) ou **toutes à la fois** (batch complet),
on utilise des **mini-batches** : groupes de N images traités simultanément.

| Mode | Avantages | Inconvénients |
|------|-----------|----------------|
| **Batch=1** (SGD pur) | Mise à jour fréquente | Bruit élevé, lent |
| **Batch=N** (full batch) | Gradient précis | Ne tient pas en mémoire |
| **Mini-batch (32–256)** | Compromis vitesse/précision | ✅ Standard |

**Batch size = 32** est la valeur la plus courante. La taille du batch affecte :
- L'utilisation mémoire GPU
- La vitesse d'entraînement
- La qualité de la généralisation (batches plus petits → meilleure généralisation)

---
# Partie 2 — Couches d'un CNN

## Conv2D — Couche de Convolution

### Théorie

La **convolution** fait glisser un petit filtre (kernel) sur l'image et calcule, à chaque position,
le **produit scalaire** entre le filtre et la zone couverte.

```
Image (5×5)          Filtre (3×3)       Feature Map (3×3)
┌───────────────┐   ┌─────────┐         ┌─────────┐
│ 1  2  3  0  1 │   │ 1  0 -1 │         │         │
│ 4  5  6  1  0 │ * │ 2  0 -2 │    →    │ valeurs │
│ 7  8  9  2  1 │   │ 1  0 -1 │         │         │
│ 0  1  2  3  4 │   └─────────┘         └─────────┘
│ 1  0  1  2  3 │
└───────────────┘
```

Chaque filtre détecte un **type de feature** :
- Filtres des premières couches → bords, gradients de couleur
- Filtres des couches profondes → formes, textures, objets

Les poids des filtres sont **appris pendant l'entraînement** (backpropagation).

### Paramètres clés

| Paramètre | Rôle |
|-----------|------|
| `filters` | Nombre de filtres (= profondeur de la sortie) |
| `kernel_size` | Taille du filtre, ex: (3,3) |
| `padding='same'` | Ajoute des zéros en bordure → sortie de même taille que l'entrée |
| `padding='valid'` | Pas de padding → la sortie est plus petite |
| `strides` | Pas de déplacement du filtre (défaut=1) |
| `activation` | Fonction d'activation appliquée à la sortie |

In [ ]:
from tensorflow.keras import layers
import tensorflow as tf

# ---- Déclaration d'une couche Conv2D ----
conv = layers.Conv2D(
    filters=32,          # 32 filtres → 32 feature maps en sortie
    kernel_size=(3, 3),  # filtre 3×3 (peut s'écrire juste 3)
    padding='same',      # 'same' : sortie même taille | 'valid' : sortie réduite
    activation='relu',   # appliquée après la convolution
    # strides=(1, 1),    # pas de déplacement (défaut)
    # use_bias=True,     # biais ajouté à chaque filtre (défaut True)
)

# Test : passer un batch fictif
x = tf.zeros((1, 128, 128, 3))  # 1 image RGB 128×128
y = conv(x)
print(f"Entrée  : {x.shape}")   # (1, 128, 128, 3)
print(f"Sortie  : {y.shape}")   # (1, 128, 128, 32)  ← 32 feature maps

# Nombre de paramètres : filters × (kernel_h × kernel_w × input_channels + 1_bias)
params = 32 * (3 * 3 * 3 + 1)
print(f"Paramètres : {params}")  # 896

## MaxPooling2D — Couche de Pooling

### Théorie

Le **pooling** réduit la résolution spatiale de la feature map en extrayant la valeur maximale
(MaxPooling) ou moyenne (AveragePooling) de chaque zone.

```
Feature map (4×4)        MaxPooling 2×2         Sortie (2×2)
┌────────────────┐                              ┌──────────┐
│  1   3   2   4 │    → prend le max de         │  3    4  │
│  5   6   1   2 │      chaque zone 2×2  →      │  8    6  │
│  7   8   3   2 │                              └──────────┘
│  4   2   1   6 │
└────────────────┘
```

**Effets :**
- Réduit la taille par un facteur `pool_size` → moins de calculs
- Crée une **invariance à la translation** (petits décalages de l'objet ignorés)
- Retient l'information la plus saillante

In [ ]:
pool = layers.MaxPooling2D(
    pool_size=(2, 2),   # fenêtre de réduction (défaut)
    strides=None,       # si None, utilise pool_size comme stride
    padding='valid',    # pas de padding (défaut)
)

x = tf.zeros((1, 64, 64, 32))   # batch après un Conv2D(32)
y = pool(x)
print(f"Entrée  : {x.shape}")   # (1, 64, 64, 32)
print(f"Sortie  : {y.shape}")   # (1, 32, 32, 32)  ← divisé par 2 en H et W

# AveragePooling2D : même syntaxe mais prend la moyenne (moins courant)
avg_pool = layers.AveragePooling2D(pool_size=(2, 2))

# GlobalAveragePooling2D : réduit toute la feature map à UN seul scalaire
# → alternative légère à Flatten en fin de réseau
gap = layers.GlobalAveragePooling2D()
x2 = tf.zeros((1, 8, 8, 256))   # feature map finale
y2 = gap(x2)
print(f"\nGAP entrée : {x2.shape}")  # (1, 8, 8, 256)
print(f"GAP sortie : {y2.shape}")   # (1, 256)  ← un vecteur de 256 features

## BatchNormalization — Normalisation par Lots

### Théorie

La **BatchNorm** normalise les activations de chaque couche pour qu'elles aient
une **moyenne ≈ 0** et un **écart-type ≈ 1** au sein de chaque mini-batch.

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \cdot \gamma + \beta$$

- $\mu_B$, $\sigma_B$ : moyenne et variance du batch
- $\gamma$, $\beta$ : paramètres **appris** qui permettent au réseau de défaire la normalisation si nécessaire

**Bénéfices :**
- Accélère la convergence (learning rate plus élevé possible)
- Réduit la sensibilité à l'initialisation des poids
- Effet régularisant (réduit légèrement l'overfitting)
- Atténue le problème de *vanishing/exploding gradients*

**Position dans un bloc CNN :** Conv → **BN** → Activation (et non Conv → Activation → BN)

In [ ]:
bn = layers.BatchNormalization(
    momentum=0.99,   # vitesse de mise à jour des statistiques (défaut 0.99)
    epsilon=1e-3,    # petite valeur pour éviter la division par zéro
)
# La couche a des paramètres entraînables (gamma, beta)
# et des paramètres non entraînables (mean, variance mobiles)

# En pratique dans un bloc Conv :
from tensorflow.keras import models
bloc = models.Sequential([
    layers.Conv2D(32, 3, padding='same'),   # pas d'activation ici
    layers.BatchNormalization(),             # normalisation
    layers.Activation('relu'),              # activation après BN
])
print("Paramètres entraînables   :", bloc.count_params())

## Dropout

### Théorie

Le **dropout** désactive aléatoirement une proportion `rate` des neurones à chaque
passage pendant l'entraînement (les neurones désactivés ont leur sortie mise à 0).

```
Sans Dropout         Avec Dropout (rate=0.5)
  ○─○─○─○             ○─○─✗─○
  ○─○─○─○    →        ○─✗─○─✗    (différent à chaque batch)
  ○─○─○─○             ✗─○─○─○
```

**Pourquoi ça marche ?**
- Empêche les neurones de **co-adapter** (dépendre les uns des autres)
- Équivaut à entraîner un **ensemble de sous-réseaux** différents
- En inférence (`training=False`), le dropout est désactivé — toutes les connexions actives

**Valeurs typiques :**
- Couches conv : `rate=0.25` (léger)
- Couches denses : `rate=0.5` (plus fort)

In [ ]:
drop = layers.Dropout(
    rate=0.5,    # 50% des neurones désactivés pendant l'entraînement
)

x = tf.ones((1, 10))   # vecteur de 10 neurones

# En entraînement : certains neurones mis à 0
y_train = drop(x, training=True)
print("Entraînement :", y_train.numpy())

# En inférence : tous les neurones actifs (dropout désactivé)
y_infer = drop(x, training=False)
print("Inférence    :", y_infer.numpy())

## Flatten vs GlobalAveragePooling2D

Ces deux couches font la **transition** entre la partie convolutive (3D) et la partie classifieur (1D).

### Flatten
Étale simplement tous les éléments en un vecteur 1D.
```
(8, 8, 256)  →  Flatten  →  (16 384,)
```
→ **Beaucoup de paramètres** dans la Dense suivante (16 384 × 512 = 8M params)  
→ Plus de risque d'overfitting

### GlobalAveragePooling2D (GAP)
Calcule la **moyenne spatiale** de chaque feature map : réduit (H, W, C) → (C,)
```
(8, 8, 256)  →  GAP  →  (256,)
```
→ **Beaucoup moins de paramètres** (256 × 512 = 131K)  
→ Meilleure généralisation  
→ Standard dans les architectures modernes (ResNet, MobileNet…)

In [ ]:
x = tf.zeros((1, 8, 8, 256))

flat = layers.Flatten()(x)
gap  = layers.GlobalAveragePooling2D()(x)

print(f"Flatten → {flat.shape}")   # (1, 16384)
print(f"GAP     → {gap.shape}")    # (1, 256)

# Impact sur la Dense suivante :
dense_after_flat = layers.Dense(512)
dense_after_gap  = layers.Dense(512)
dense_after_flat.build(flat.shape)
dense_after_gap.build(gap.shape)
print(f"\nParams Dense après Flatten : {dense_after_flat.count_params():,}")
print(f"Params Dense après GAP     : {dense_after_gap.count_params():,}")

## Dense — Couche Entièrement Connectée

### Théorie

Chaque neurone est **connecté à tous les neurones** de la couche précédente.
$$y = \text{activation}(xW + b)$$

- $x$ : vecteur d'entrée
- $W$ : matrice de poids (appris)
- $b$ : biais (appris)

**Usage dans un CNN :**
- **Couche(s) intermédiaire(s)** → combinent les features pour la classification (avec ReLU)
- **Couche finale** → autant de neurones que de classes, activation Softmax

In [ ]:
# Couche intermédiaire
dense_hidden = layers.Dense(
    units=512,           # nombre de neurones
    activation='relu',   # fonction d'activation
    # kernel_regularizer=tf.keras.regularizers.l2(1e-4)  # régularisation L2 (optionnel)
)

# Couche de sortie — classification 5 classes
dense_output = layers.Dense(
    units=5,             # un neurone par classe
    activation='softmax' # probabilités (somme = 1)
)

x = tf.zeros((1, 256))
print(f"Entrée  : {x.shape}")
print(f"Hidden  : {dense_hidden(x).shape}")   # (1, 512)
print(f"Output  : {dense_output(dense_hidden(x)).shape}")  # (1, 5)

---
# Partie 3 — Fonctions d'Activation

## ReLU — Rectified Linear Unit

$$\text{ReLU}(x) = \max(0, x)$$

- Toutes les valeurs négatives → 0, valeurs positives → inchangées
- **Standard** dans les couches cachées des CNNs
- Simple et rapide à calculer
- Atténue le problème de vanishing gradient (comparé à sigmoid/tanh)

---

## Softmax

$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

- Convertit un vecteur de logits (scores bruts) en **probabilités** (somme = 1)
- Utilisée **uniquement en couche de sortie** pour la classification multi-classes
- La classe prédite = celle avec la probabilité maximale

---

## Sigmoid

$$\sigma(x) = \frac{1}{1 + e^{-x}} \in ]0, 1[$$

- Comprime toute valeur dans [0, 1]
- Utilisée pour la **classification binaire** (1 neurone de sortie)
- Peu utilisée dans les couches cachées (risque de vanishing gradient)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-5, 5, 200)

relu    = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))
softmax_demo = np.array([0.1, 2.0, 0.5, -1.0, 1.5])
softmax_out  = np.exp(softmax_demo) / np.exp(softmax_demo).sum()

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

ax1.plot(x, relu, 'b', lw=2)
ax1.set_title('ReLU : max(0, x)', fontsize=12)
ax1.axhline(0, color='k', lw=0.5); ax1.axvline(0, color='k', lw=0.5)
ax1.grid(alpha=0.3)

ax2.plot(x, sigmoid, 'r', lw=2)
ax2.set_title('Sigmoid : 1/(1+e^-x)', fontsize=12)
ax2.axhline(0.5, color='k', ls='--', lw=0.5)
ax2.grid(alpha=0.3)

ax3.bar(range(5), softmax_out, color='#3498db')
ax3.set_title(f'Softmax — somme={softmax_out.sum():.4f}', fontsize=12)
ax3.set_xlabel('Classe'); ax3.set_ylabel('Probabilité')
ax3.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Softmax input  : {softmax_demo}")
print(f"Softmax output : {softmax_out.round(3)}")
print(f"Somme          : {softmax_out.sum():.6f}")

---
# Partie 4 — Apprentissage

## Loss Function — Fonction de Perte

La **loss** mesure l'écart entre les **prédictions du modèle** et les **vraies étiquettes**.
L'entraînement consiste à **minimiser** cette valeur.

### Categorical Crossentropy

$$\mathcal{L} = -\sum_{i} y_i \log(\hat{y}_i)$$

- $y_i$ : vrai label (one-hot : [0, 0, 1, 0, 0])
- $\hat{y}_i$ : probabilité prédite par le modèle
- Pénalise fortement les prédictions sûres mais fausses (ex: prédire 95% pour la mauvaise classe)

### Sparse Categorical Crossentropy

**Identique**, mais accepte les labels sous forme d'**entiers** (0, 1, 2…) au lieu de one-hot.  
→ Plus pratique quand les labels sont des indices de classes.

```python
# Categorical (labels one-hot)  : y_true = [[0, 0, 1, 0, 0]]
# Sparse categorical (labels int): y_true = [2]
```

In [ ]:
from tensorflow.keras import losses

# Prédictions softmax (probabilités)
y_pred = tf.constant([[0.05, 0.05, 0.80, 0.05, 0.05]])  # classe 2 prédite avec 80%

# ---- Sparse Categorical (labels entiers) ----
y_true_sparse = tf.constant([2])  # vrai label = classe 2
loss_fn = losses.SparseCategoricalCrossentropy()
loss_val = loss_fn(y_true_sparse, y_pred)
print(f"Bonne prédiction (80%) → loss : {loss_val:.4f}")  # petit

# Mauvaise prédiction
y_pred_bad = tf.constant([[0.80, 0.05, 0.05, 0.05, 0.05]])  # classe 0 prédite
loss_bad   = loss_fn(y_true_sparse, y_pred_bad)
print(f"Mauvaise prédiction     → loss : {loss_bad:.4f}")   # grand

# Rappel : -log(0.80) ≈ 0.22, -log(0.05) ≈ 3.0

## Optimizer — Optimiseur

L'optimiseur détermine **comment mettre à jour les poids** à partir du gradient de la loss.

### Descente de Gradient (SGD)
$$W \leftarrow W - \alpha \cdot \nabla_W \mathcal{L}$$
- $\alpha$ : learning rate
- Simple mais peut osciller ou converger lentement

### Adam (Adaptive Moment Estimation) ✅ Recommandé
Combine **momentum** (inertie du gradient) et **RMSProp** (adaptation du LR par paramètre) :
- S'adapte automatiquement à chaque paramètre
- Converge plus vite que SGD en pratique
- Peu sensible au choix du learning rate initial
- **Standard** pour les CNNs

### Learning Rate (LR)
Contrôle la **taille des pas** lors de la mise à jour des poids :
- Trop grand → oscille, diverge
- Trop petit → converge très lentement
- **Valeur typique pour Adam** : `1e-3` au départ, réduit progressivement

In [ ]:
from tensorflow import keras

# Adam (recommandé)
adam = keras.optimizers.Adam(
    learning_rate=1e-3,  # 0.001 — valeur standard
    beta_1=0.9,          # momentum (défaut)
    beta_2=0.999,        # adaptation du LR (défaut)
    epsilon=1e-7,        # stabilité numérique (défaut)
)

# SGD avec momentum
sgd = keras.optimizers.SGD(
    learning_rate=1e-2,
    momentum=0.9,        # inertie du gradient
    nesterov=True,       # variante améliorée
)

print("Adam  :", adam.get_config()['learning_rate'])
print("SGD   :", sgd.get_config()['learning_rate'])

## Epoch / Batch / Step

| Terme | Définition |
|-------|------------|
| **Step** (itération) | Un passage forward + backward sur **un batch** |
| **Epoch** | Un passage complet sur **tout le dataset** d'entraînement |
| **Steps per epoch** | Nombre de batches par epoch = ⌈N_images / batch_size⌉ |

**Exemple avec 33 120 images et batch_size=32 :**
```
Steps per epoch = 33 120 / 32 = 1 035 steps
Epoch 1 : steps 1 → 1 035  (vu toutes les images)
Epoch 2 : steps 1 → 1 035  (ré-vu toutes les images, dans un ordre différent)
...
```

**Nombre d'epochs :** en pratique, on utilise **EarlyStopping** pour arrêter
automatiquement quand la validation ne s'améliore plus.

## Overfitting / Underfitting — Biais / Variance

```
 Accuracy
    │
 1  │  train ────────────────────────────────
    │  val   ──────────────────────────────── ← Bon équilibre ✅
    │
    │  train ────────────────────────────
    │  val   ─────────────                 ← Surapprentissage (écart croissant)
    │
 0  │  train ──────────                    ← Sous-apprentissage (les deux basses)
    │  val   ──────────
    └──────────────────────────────────► Epochs
```

### Sous-apprentissage (Underfitting) — Biais élevé
- Train accuracy **et** val accuracy sont toutes deux basses
- Modèle trop simple, pas assez de capacité, trop peu d'epochs
- **Solutions** : architecture plus profonde, plus d'epochs, LR adapté

### Surapprentissage (Overfitting) — Variance élevée
- Train accuracy haute, val accuracy bien inférieure (grand écart)
- Modèle mémorise les données d'entraînement, ne généralise pas
- **Solutions** : Dropout, data augmentation, BatchNorm, EarlyStopping, régularisation L2

---
# Partie 5 — Techniques de Régularisation

## Data Augmentation — Augmentation des Données

### Théorie

Génère des **variantes d'images existantes** en appliquant des transformations aléatoires.
Chaque epoch, le modèle voit des images légèrement différentes → il ne peut pas les mémoriser.

**Transformations courantes :**

| Transformation | Quand l'utiliser |
|----------------|------------------|
| `RandomFlip` | Toujours (sauf si l'orientation a un sens : ex. chiffres) |
| `RandomRotation` | Objets pouvant être à n'importe quel angle |
| `RandomZoom` | Objets à distances variables |
| `RandomBrightness` | Conditions d'éclairage variables |
| `RandomContrast` | Variabilité photo |

**Important :** l'augmentation s'applique **uniquement sur le train**, jamais sur val/test.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),      # retournement horizontal (miroir)
    layers.RandomRotation(0.10),          # rotation ±10% de 360° = ±36°
    layers.RandomZoom(0.10),              # zoom ±10%
    layers.RandomBrightness(factor=0.15), # luminosité ±15%
    layers.RandomContrast(factor=0.15),   # contraste ±15%
], name="data_augmentation")

# L'augmentation est désactivée en inférence (training=False)
img = tf.zeros((1, 128, 128, 3))
aug = data_augmentation(img, training=True)
print(f"Entrée : {img.shape} → Sortie : {aug.shape}")
print("La shape ne change pas, mais les valeurs sont modifiées")

## Class Weights — Poids des Classes

### Théorie

Quand les classes sont **déséquilibrées** (ex: 8 000 images d'une classe vs 1 000 d'une autre),
le modèle tend à privilégier les classes majoritaires.

Les **class weights** compensent ce déséquilibre en **augmentant la pénalité** (loss) pour
les erreurs sur les classes rares :

$$w_i = \frac{N_{total}}{N_{classes} \times N_i}$$

- Classe rare → poids élevé (chaque erreur coûte plus)
- Classe abondante → poids faible

**Exemple :** si Sketch a 1 148 images et les autres ~8 000 :
$$w_{Sketch} = \frac{33 120}{5 \times 1 148} \approx 5.77$$

In [ ]:
import numpy as np

# Compte d'images par classe (exemple projet TouNum)
label_counts = np.array([8003, 7971, 7940, 1148, 8058])
CLASS_NAMES  = ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
n_total      = label_counts.sum()
n_classes    = len(CLASS_NAMES)

class_weights = {
    i: n_total / (n_classes * count)
    for i, count in enumerate(label_counts)
}

print(f"{'Classe':<15} {'Images':>8} {'Weight':>9}")
print("-" * 35)
for i, (cls, cnt) in enumerate(zip(CLASS_NAMES, label_counts)):
    print(f"{cls:<15} {cnt:>8,} {class_weights[i]:>9.3f}")

# Utilisation : model.fit(..., class_weight=class_weights)

## Callbacks

Les **callbacks** sont des fonctions appelées automatiquement par Keras **à chaque epoch**
(ou batch) pendant `model.fit()`. Ils permettent de surveiller et d'adapter l'entraînement.

### EarlyStopping
Arrête l'entraînement si la métrique surveillée ne s'améliore plus pendant N epochs.
- `monitor` : métrique à surveiller (`'val_loss'` ou `'val_accuracy'`)
- `patience` : nombre d'epochs à attendre avant d'arrêter
- `restore_best_weights` : recharge les poids du meilleur epoch

### ReduceLROnPlateau
Réduit le learning rate quand la métrique plafonne.
- `factor` : multiplicateur du LR (ex: `0.5` = divise par 2)
- `patience` : epochs à attendre avant de réduire

### ModelCheckpoint
Sauvegarde le modèle quand il atteint un nouveau meilleur score.
- `save_best_only=True` : ne sauvegarde que si amélioration
- `monitor` : métrique à surveiller pour décider de sauvegarder

In [ ]:
from tensorflow import keras

callbacks = [
    # Arrêt anticipé
    keras.callbacks.EarlyStopping(
        monitor='val_loss',          # surveille la perte de validation
        patience=5,                  # arrête si pas d'amélioration pendant 5 epochs
        restore_best_weights=True,   # revient aux meilleurs poids
        verbose=1,
    ),

    # Réduction du learning rate
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,       # nouveau LR = LR × 0.5 (divise par 2)
        patience=3,       # déclenche après 3 epochs sans amélioration
        min_lr=1e-7,      # plancher — ne descend pas en dessous
        verbose=1,
    ),

    # Sauvegarde du meilleur modèle
    keras.callbacks.ModelCheckpoint(
        filepath='./model_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_format='h5',   # obligatoire sous TF 2.14 (bug avec .keras)
        verbose=1,
    ),
]

print("Callbacks prêts :", [cb.__class__.__name__ for cb in callbacks])

---
# Partie 6 — Pipeline tf.data

## tf.data — Pipeline de Données

`tf.data.Dataset` est l'API TensorFlow pour construire des **pipelines de chargement et
prétraitement de données** efficaces. Les opérations sont lazily évaluées (exécutées
au moment de l'itération, pas à la création).

### Opérations principales

| Opération | Rôle |
|-----------|------|
| `.map(fn)` | Applique une fonction à chaque élément (prétraitement, augmentation) |
| `.batch(n)` | Groupe les éléments en batches de taille n |
| `.shuffle(buffer)` | Mélange aléatoirement (buffer = fenêtre de mélange) |
| `.prefetch(n)` | Prépare les prochains batches pendant le calcul du batch courant |
| `.cache()` | Stocke le dataset en RAM après le 1er passage |
| `.repeat()` | Répète le dataset infiniment (utile avec steps_per_epoch) |
| `.take(n)` | Garde uniquement les n premiers éléments |
| `.skip(n)` | Saute les n premiers éléments |
| `.filter(fn)` | Garde uniquement les éléments pour lesquels fn retourne True |

### Ordre recommandé
```
dataset = dataset.shuffle().map(preprocess).batch().prefetch()
```
Le `.shuffle()` avant `.batch()` mélange les exemples individuels (meilleur mélange).

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE  # TF choisit automatiquement le parallélisme optimal

# ---- image_dataset_from_directory ----
# Crée un Dataset à partir d'un dossier organisé par sous-dossiers (= classes)
# Syntaxe :
# tf.keras.utils.image_dataset_from_directory(
#     directory,              # chemin vers le dossier racine
#     validation_split=0.2,   # fraction pour validation
#     subset='training',      # 'training' ou 'validation'
#     seed=42,                # reproductibilité
#     image_size=(128, 128),  # redimensionnement
#     batch_size=32,          # taille des batches
#     label_mode='int',       # labels entiers (0, 1, 2…)
#     shuffle=True,           # mélanger les fichiers
# )

# ---- Exemple de pipeline complet ----
# (simulation avec un dataset factice)

images = tf.random.uniform((100, 128, 128, 3), 0, 255)
labels = tf.random.uniform((100,), 0, 5, dtype=tf.int32)
dataset = tf.data.Dataset.from_tensor_slices((images, labels))

def preprocess(img, lbl):
    img = tf.cast(img, tf.float32) / 255.0   # normalisation
    return img, lbl

pipeline = (
    dataset
    .shuffle(buffer_size=1000)                        # mélange dans une fenêtre de 1000 éléments
    .map(preprocess, num_parallel_calls=AUTOTUNE)     # prétraitement parallèle
    .batch(32)                                        # grouper en batches
    .prefetch(AUTOTUNE)                               # préparer le batch suivant en avance
)

for batch_img, batch_lbl in pipeline.take(1):
    print(f"Batch images : {batch_img.shape}")
    print(f"Batch labels : {batch_lbl.shape}")
    print(f"Valeurs normalisées : [{batch_img.numpy().min():.3f}, {batch_img.numpy().max():.3f}]")

---
# Partie 7 — API Keras : compiler, entraîner, évaluer

## model.compile()

Configure le modèle pour l'entraînement. Doit être appelé **avant** `model.fit()`.

```python
model.compile(
    optimizer,    # algorithme d'optimisation
    loss,         # fonction de perte
    metrics,      # métriques à afficher (pas utilisées pour optimiser)
)
```

## model.fit()

Lance l'entraînement. Retourne un objet `history` contenant l'évolution des métriques.

```python
history = model.fit(
    train_ds,              # dataset (ou numpy array) d'entraînement
    epochs=30,             # nombre maximum d'epochs
    validation_data=val_ds,# dataset de validation (jamais utilisé pour optimiser)
    class_weight={...},    # poids des classes (optionnel)
    callbacks=[...],       # liste de callbacks
    verbose=1,             # 0=silencieux, 1=barre de progression, 2=une ligne/epoch
)
```

**Résultat — `history.history` :**
```python
{
    'loss':         [0.75, 0.55, 0.45, ...],
    'accuracy':     [0.64, 0.73, 0.80, ...],
    'val_loss':     [0.72, 0.69, 0.55, ...],
    'val_accuracy': [0.74, 0.73, 0.81, ...],
    'lr':           [0.001, 0.001, 0.0005, ...],
}
```

## model.evaluate() et model.predict()

### evaluate()
Calcule la loss et les métriques sur un dataset. Retourne `[loss, accuracy]`.
```python
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
```

### predict()
Retourne les **probabilités softmax** pour chaque image, sans calculer la loss.
```python
probs = model.predict(images)   # shape (N, n_classes)
classes = np.argmax(probs, axis=1)  # classe avec la probabilité max
```

### Différence clé
| | `evaluate` | `predict` |
|--|--|--|
| **Besoin des labels** | Oui | Non |
| **Calcule la loss** | Oui | Non |
| **Retourne** | [loss, metric] | probabilités |
| **Usage** | Mesurer les performances | Faire des prédictions |

In [ ]:
import numpy as np
import tensorflow as tf

# Simulation de prédictions softmax
probs = np.array([
    [0.02, 0.03, 0.80, 0.10, 0.05],  # image 1 → classe 2 (Schematics)
    [0.70, 0.15, 0.05, 0.05, 0.05],  # image 2 → classe 0 (Painting)
    [0.05, 0.90, 0.02, 0.01, 0.02],  # image 3 → classe 1 (Photo)
])

CLASS_NAMES = ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']

# Classe prédite = indice de la probabilité maximale
pred_classes = np.argmax(probs, axis=1)
pred_names   = [CLASS_NAMES[i] for i in pred_classes]
confidences  = probs.max(axis=1) * 100

print(f"{'Image':<8} {'Classe prédite':<15} {'Confiance':>10}")
print("-" * 36)
for i, (cls, conf) in enumerate(zip(pred_names, confidences)):
    print(f"Image {i+1}  {cls:<15} {conf:>9.1f}%")

---
# Partie 8 — Évaluation du Modèle

## Accuracy, Précision, Rappel, F1-score

### Matrice de confusion

Pour une classe donnée, 4 cas possibles :

|  | Prédit Positif | Prédit Négatif |
|--|--|--|
| **Réel Positif** | TP (Vrai Positif) | FN (Faux Négatif) |
| **Réel Négatif** | FP (Faux Positif) | TN (Vrai Négatif) |

### Métriques

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{\text{corrections}}{\text{total}}$$

$$\text{Précision} = \frac{TP}{TP + FP} \quad \text{(parmi ceux prédits positifs, combien le sont vraiment ?)}$$

$$\text{Rappel} = \frac{TP}{TP + FN} \quad \text{(parmi les vrais positifs, combien sont détectés ?)}$$

$$\text{F1-score} = 2 \times \frac{\text{Précision} \times \text{Rappel}}{\text{Précision} + \text{Rappel}} \quad \text{(moyenne harmonique)}$$

**Quand utiliser quoi ?**
- **Accuracy** : quand les classes sont équilibrées
- **Rappel** : quand les faux négatifs sont coûteux (ex: diagnostic médical)
- **Précision** : quand les faux positifs sont coûteux (ex: spam)
- **F1** : compromis — à privilégier avec des classes déséquilibrées

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

CLASS_NAMES = ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
n_cls = len(CLASS_NAMES)

# Simulation d'une matrice de confusion réaliste
cm = np.array([
    [720, 80,  10,  5,  15],  # Painting  : souvent confondu avec Photo
    [60,  750, 5,   2,  5],   # Photo
    [5,   3,   800, 2,  10],  # Schematics
    [10,  5,   3,   80, 7],   # Sketch    (peu d'images)
    [5,   2,   15,  1,  800], # Text
])

# --- Visualisation ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title('Matrice de confusion (valeurs absolues)')
ax1.set_xlabel('Classe prédite'); ax1.set_ylabel('Classe réelle')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
ax2.set_title('Rappel normalisé (par ligne)')
ax2.set_xlabel('Classe prédite'); ax2.set_ylabel('Classe réelle')

plt.tight_layout()
plt.show()

# --- Rapport de classification ---
print(f"\n{'Classe':<15} {'Précision':>10} {'Rappel':>9} {'F1-score':>10} {'Support':>9}")
print("-" * 60)
for i, cls in enumerate(CLASS_NAMES):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    support   = int(cm[i, :].sum())
    print(f"{cls:<15} {precision:>10.3f} {recall:>9.3f} {f1:>10.3f} {support:>9,}")
print("-" * 60)
print(f"{'Accuracy':>15} {cm.diagonal().sum() / cm.sum():>10.3f}")

---
# Partie 9 — Transfer Learning

### Théorie

Plutôt qu'entraîner un CNN from scratch, le **Transfer Learning** réutilise
un modèle **pré-entraîné sur ImageNet** (1,4M images, 1 000 classes).

Les premières couches ont déjà appris des features génériques (bords, textures, formes)
applicables à n'importe quel domaine. On les réutilise directement ou on les ajuste.

### Feature Extraction vs Fine-Tuning

| Mode | Description | Quand l'utiliser |
|------|-------------|------------------|
| **Feature Extraction** | Base gelée, seule la tête est entraînée | Dataset petit, tâche proche d'ImageNet |
| **Fine-Tuning partiel** | Dégel des dernières couches de la base | Dataset moyen, domaine légèrement différent |
| **Fine-Tuning complet** | Toutes les couches entraînées (LR très faible) | Grand dataset, domaine très différent |

```
Couches profondes  (gelées)  → features bas niveau : bords, coins, gradients
Couches hautes     (à dégeler progressivement)     → textures, formes spécifiques
Nouvelle tête      (toujours entraînée)            → classifieur de notre tâche
```

### Architectures disponibles dans Keras

| Modèle | Params | Top-1 ImageNet | Usage recommandé |
|--------|--------|----------------|-----------------|
| **MobileNetV2** | 3.4M | 71.8% | Mobile, temps réel, peu de VRAM |
| **EfficientNetB0** | 5.3M | 77.1% | Meilleur compromis taille/précision |
| **EfficientNetB4** | 19M | 83.0% | Haute précision si VRAM suffisante |
| **ResNet50** | 25M | 76.0% | Référence académique solide |
| **InceptionV3** | 24M | 78.8% | Features multi-échelle |
| **VGG16** | 138M | 73.0% | Simple mais lourd — legacy |

### Stratégie de learning rate pour le fine-tuning

Lors du dégel, le LR doit être **10× à 100× plus faible** que pour la tête.
Sinon on détruit les features pré-entraînées (catastrophic forgetting).

```
Étape 1 — Feature extraction   : LR = 1e-3  (tête seule)
Étape 2 — Fine-tuning partiel  : LR = 1e-5  (dernières couches + tête)
Étape 3 — Fine-tuning complet  : LR = 1e-6  (tout le réseau)
```

### Prétraitement spécifique à chaque architecture

Chaque modèle a été entraîné avec une normalisation différente. Utiliser
`tf.keras.applications.<Modele>.preprocess_input(x)` avant de passer les images.

In [ ]:
from tensorflow.keras import layers
import tensorflow as tf

IMG_SIZE    = (224, 224)
NUM_CLASSES = 5

# ============================================================
# ÉTAPE 1 — Feature Extraction (base gelée, tête entraînée)
# ============================================================
base = tf.keras.applications.EfficientNetB0(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,      # supprime la tête ImageNet (Dense 1000)
    weights=None,           # mettre 'imagenet' en pratique
    pooling='avg',          # GlobalAveragePooling2D intégré → sortie (batch, 1280)
)
base.trainable = False      # ← gèle toutes les couches

inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
x       = tf.keras.applications.efficientnet.preprocess_input(inputs)
x       = base(x, training=False)  # training=False : BN en mode inférence
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model   = tf.keras.Model(inputs, outputs, name='EfficientNetB0_FeatureExtraction')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

trainable = sum(v.numpy().size for v in model.trainable_variables)
frozen    = sum(v.numpy().size for v in model.non_trainable_variables)
print("=== Feature Extraction ===")
print(f"Params entraînables : {trainable:,}")
print(f"Params gelés (base) : {frozen:,}")
print(f"Ratio gelé          : {frozen/(trainable+frozen)*100:.1f}%")

# ============================================================
# ÉTAPE 2 — Fine-Tuning (dégel des N dernières couches)
# ============================================================
base.trainable = True

FINE_TUNE_FROM = len(base.layers) - 20   # dégèle les 20 dernières couches
for layer in base.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

n_trainable = sum(1 for l in base.layers if l.trainable)
print(f"\n=== Fine-Tuning ===")
print(f"Couches dégelées dans la base : {n_trainable}/{len(base.layers)}")

# LR 100× plus faible pour ne pas écraser les features pré-entraînées
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# En pratique :
# history_ft = model.fit(train_ds, epochs=10, validation_data=val_ds,
#                        callbacks=[early_stop, reduce_lr])

print("Recompilé avec LR=1e-5 — prêt pour le fine-tuning")

# ============================================================
# PRÉTRAITEMENT selon le modèle choisi
# ============================================================
preprocessing_map = {
    'EfficientNetB0'  : 'tf.keras.applications.efficientnet.preprocess_input',
    'MobileNetV2'     : 'tf.keras.applications.mobilenet_v2.preprocess_input',
    'ResNet50'        : 'tf.keras.applications.resnet.preprocess_input',
    'InceptionV3'     : 'tf.keras.applications.inception_v3.preprocess_input',
    'VGG16'           : 'tf.keras.applications.vgg16.preprocess_input',
}
print("\nPrétraitements :")
for model_name, fn in preprocessing_map.items():
    print(f"  {model_name:<20} → {fn}")

---
# Partie 10 — Autoencodeurs & VAE

## Autoencodeur — Architecture et Théorie

Un **autoencodeur** est un réseau qui apprend à **compresser** une entrée
vers une représentation compacte (espace latent), puis à la **reconstruire**.

### Structure

```
         ENCODEUR                        DÉCODEUR
   Input (28x28x1)  ──►  z (64d)  ──►  Output (28x28x1)
   Conv → MaxPool                       UpSamp → Conv
   Conv → MaxPool                       UpSamp → Conv
   Flatten → Dense                      Dense → Reshape
```

| Composant | Rôle |
|-----------|------|
| **Encodeur** | Compresse x vers z |
| **Espace latent (bottleneck)** | Représentation compacte des features |
| **Décodeur** | Reconstruit x̂ depuis z |

### Reconstruction Loss

- **MSE** : pour pixels continus → `loss = mean((x - x_hat)^2)`
- **BCE** : pour pixels binaires → `loss = binary_crossentropy(x, x_hat)`

### Applications

| Application | Principe |
|-------------|----------|
| **Compression** | Stocker z plutôt que x |
| **Débruitage** | Entrée bruitée, cible propre |
| **Détection d'anomalies** | Grosse erreur de reconstruction = anomalie |
| **Pré-entraînement** | L'encodeur initialise un classifieur |

### Autoencodeur Débruiteur

```
x propre  →  [+ bruit]  →  x̃  →  Encodeur  →  z  →  Décodeur  →  x̂ ≈ x propre
```

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# ============================================================
# AUTOENCODEUR CONVOLUTIF (28x28 niveaux de gris)
# ============================================================

# ---- Encodeur ----
enc_in  = tf.keras.Input(shape=(28, 28, 1))
x       = layers.Conv2D(32, 3, activation='relu', padding='same')(enc_in)   # (28,28,32)
x       = layers.MaxPooling2D(2)(x)                                          # (14,14,32)
x       = layers.Conv2D(64, 3, activation='relu', padding='same')(x)        # (14,14,64)
x       = layers.MaxPooling2D(2)(x)                                          # (7,7,64)
x       = layers.Flatten()(x)
latent  = layers.Dense(64, name='latent')(x)
encoder = tf.keras.Model(enc_in, latent, name='Encodeur')

# ---- Decodeur ----
dec_in  = tf.keras.Input(shape=(64,))
x       = layers.Dense(7 * 7 * 64, activation='relu')(dec_in)
x       = layers.Reshape((7, 7, 64))(x)
x       = layers.Conv2DTranspose(64, 3, activation='relu', padding='same')(x)
x       = layers.UpSampling2D(2)(x)
x       = layers.Conv2DTranspose(32, 3, activation='relu', padding='same')(x)
x       = layers.UpSampling2D(2)(x)
dec_out = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
decoder = tf.keras.Model(dec_in, dec_out, name='Decodeur')

# ---- Autoencodeur complet ----
ae_in  = tf.keras.Input(shape=(28, 28, 1))
ae_out = decoder(encoder(ae_in))
ae     = tf.keras.Model(ae_in, ae_out, name='Autoencoder')
ae.compile(optimizer='adam', loss='binary_crossentropy')

print(f"Pixels image         : {28*28}")
print(f"Dimension latente    : 64")
print(f"Taux de compression  : {28*28/64:.1f}x")
print(f"Params encodeur      : {encoder.count_params():,}")
print(f"Params decodeur      : {decoder.count_params():,}")
print()
print("Entrainement standard  : ae.fit(x_train, x_train, ...)")
print("Entrainement debruiteur: ae.fit(x_noisy, x_clean, ...)")

# Demo
x_test  = np.random.rand(4, 28, 28, 1).astype('float32')
z_codes = encoder.predict(x_test, verbose=0)
x_recon = decoder.predict(z_codes, verbose=0)
print(f"\nEntree : {x_test.shape}  →  Code z : {z_codes.shape}  →  Sortie : {x_recon.shape}")

## Autoencodeur Variationnel (VAE)

### Problème des AE classiques pour la génération

Un AE classique mappe vers un **point fixe** — l'espace latent n'est pas structuré.
Impossible de savoir quels z génèrent des images valides.

### Solution : espace latent probabiliste

Le VAE mappe vers une **distribution** N(µ, σ²). On échantillonne depuis cette distribution :

```
      Entree x
           |
       Encodeur
          / \
       mu(x)  log_sigma2(x)    <- deux tetes Dense
          \ /
    z = mu + sigma * epsilon    <- reparameterization trick : epsilon ~ N(0,1)
           |                      (permet la retropropagation !)
       Decodeur
           |
         x_hat ~= x
```

### Reparameterization Trick

On ne peut pas derivé à travers un noeud d'échantillonnage aléatoire.
Astuce : **z = µ + σ · ε** avec ε ~ N(0,1) tiré indépendamment.
Le gradient passe normalement par µ et σ.

### Fonction de perte ELBO

```
L_VAE = Reconstruction Loss + beta * KL Divergence

  Reconstruction : || x - x_hat ||^2  (le décodeur doit bien reconstruire)
  KL divergence  : -0.5 * sum(1 + log_sigma^2 - mu^2 - sigma^2)
                   (force q(z|x) ~ N(0,1) -> espace latent continu)
  beta           : coefficient d'équilibre (beta-VAE)
```

### Génération et Interpolation

```python
# Generation : z ~ N(0,I) -> nouvelle image sans encodeur
z_sample = tf.random.normal(shape=(n, LATENT_DIM))
images   = decoder(z_sample)

# Interpolation entre deux images A et B
z_interp = (1-alpha)*z_A + alpha*z_B  # alpha in [0,1]
```

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

LATENT_DIM = 32

# ---- Couche Sampling (reparameterization trick) ----
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps

# ---- Encodeur : x -> (mu, log_sigma2, z) ----
enc_in    = tf.keras.Input(shape=(28, 28, 1))
x         = layers.Conv2D(32, 3, activation='relu', padding='same')(enc_in)
x         = layers.MaxPooling2D(2)(x)
x         = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x         = layers.MaxPooling2D(2)(x)
x         = layers.Flatten()(x)
x         = layers.Dense(128, activation='relu')(x)
z_mean    = layers.Dense(LATENT_DIM, name='z_mean')(x)
z_log_var = layers.Dense(LATENT_DIM, name='z_log_var')(x)
z         = Sampling()([z_mean, z_log_var])
encoder   = tf.keras.Model(enc_in, [z_mean, z_log_var, z], name='VAE_Encoder')

# ---- Decodeur : z -> x_hat ----
dec_in  = tf.keras.Input(shape=(LATENT_DIM,))
x       = layers.Dense(7 * 7 * 64, activation='relu')(dec_in)
x       = layers.Reshape((7, 7, 64))(x)
x       = layers.Conv2DTranspose(64, 3, activation='relu', padding='same')(x)
x       = layers.UpSampling2D(2)(x)
x       = layers.Conv2DTranspose(32, 3, activation='relu', padding='same')(x)
x       = layers.UpSampling2D(2)(x)
dec_out = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
decoder = tf.keras.Model(dec_in, dec_out, name='VAE_Decoder')

# ---- VAE avec ELBO loss ----
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, beta=1.0, **kw):
        super().__init__(**kw)
        self.encoder = encoder
        self.decoder = decoder
        self.beta    = beta

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x, training=True)
            x_hat = self.decoder(z, training=True)
            recon = tf.reduce_mean(
                tf.reduce_sum(tf.keras.losses.binary_crossentropy(x, x_hat), axis=(1,2)))
            kl = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            loss = recon + self.beta * kl
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {'loss': loss, 'reconstruction': recon, 'kl': kl}

vae = VAE(encoder, decoder, beta=1.0)
vae.compile(optimizer='adam')

print(f"Espace latent   : {LATENT_DIM}D")
print(f"Params encodeur : {encoder.count_params():,}")
print(f"Params decodeur : {decoder.count_params():,}")

# Generation
z_rand    = tf.random.normal(shape=(6, LATENT_DIM))
generated = decoder(z_rand)
print(f"\nImages generees (sans encodeur) : {generated.shape}")

# Interpolation
z1 = tf.random.normal((1, LATENT_DIM))
z2 = tf.random.normal((1, LATENT_DIM))
z_interp = tf.stack([(1-a)*z1[0] + a*z2[0] for a in np.linspace(0,1,8)])
interp   = decoder(z_interp)
print(f"Interpolation z1->z2            : {interp.shape}")

---
# Partie 11 — RNN & Transformers

## RNN — Réseau de Neurones Récurrent

### Pourquoi pas les CNN pour les séquences ?

Les CNN traitent à **taille fixe** sans ordre temporel.
Pour le texte, audio, vidéo : l'ordre compte.

### Architecture

```
x1 -> [RNN] -> [RNN] -> [RNN] -> [RNN] -> sortie
        h1      h2      h3      h4      <- état caché (mémoire)
```

```
h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)
```

Les poids W_h, W_x sont **partagés** à chaque pas de temps.

### Vanishing Gradient

La rétropropagation multiplie le gradient à chaque pas.
Sur de longues séquences, le gradient **disparaît** (< 1) ou **explose** (> 1).
→ Le RNN simple **oublie** les informations lointaines.

**Solution** : LSTM et GRU.

## LSTM — Long Short-Term Memory

Résout le vanishing gradient avec une **cell state** (mémoire long terme)
et 3 **portes** qui contrôlent le flux d'information.

### Les 3 portes

| Porte | Rôle |
|-------|------|
| **Forget gate** f_t = σ(...) | Qu'oublier de la cell state |
| **Input gate** i_t = σ(...) | Quoi écrire dans la cell state |
| **Output gate** o_t = σ(...) | Quoi lire depuis la cell state |

```
C_t = f_t * C_{t-1} + i_t * tanh(W_c * [h_{t-1}, x_t])   # cell state
h_t = o_t * tanh(C_t)                                       # hidden state
```

- **C_t** : mémoire long terme (cell state)
- **h_t** : sortie (mémoire court terme)

## GRU — Gated Recurrent Unit

Version simplifiée du LSTM : **2 portes** (reset + update).
Moins de paramètres, convergence similaire.

```python
lstm = layers.LSTM(128, return_sequences=True)    # return_seq pour stacker
gru  = layers.GRU(128, return_sequences=False)    # False = sortie finale seulement
bilstm = layers.Bidirectional(layers.LSTM(64))    # lecture dans les 2 sens
```

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

SEQ_LEN   = 50
N_FEAT    = 64
N_CLASSES = 5

# ---- LSTM bidirectionnel ----
model_lstm = tf.keras.Sequential([
    layers.Input(shape=(SEQ_LEN, N_FEAT)),
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(N_CLASSES, activation='softmax'),
], name='BiLSTM')

model_lstm.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
model_lstm.summary()

# ---- GRU ----
model_gru = tf.keras.Sequential([
    layers.Input(shape=(SEQ_LEN, N_FEAT)),
    layers.GRU(128, return_sequences=True),
    layers.GRU(64),
    layers.Dense(N_CLASSES, activation='softmax'),
], name='GRU')

x_seq = np.random.randn(8, SEQ_LEN, N_FEAT).astype('float32')
print(f'Entree   : {x_seq.shape}')
print(f'Sortie LSTM : {model_lstm.predict(x_seq, verbose=0).shape}')
print(f'Sortie GRU  : {model_gru.predict(x_seq, verbose=0).shape}')
print(f'Params LSTM : {model_lstm.count_params():,}')
print(f'Params GRU  : {model_gru.count_params():,}')

## Mécanisme d'Attention

### Intuition

L'attention permet à chaque élément de regarder **tous les autres** et de pondérer
leur importance. Pour traiter `"mange"`, on se concentre sur `"chat"` (sujet).

### Scaled Dot-Product Attention

Chaque token produit 3 vecteurs :

| Vecteur | Signification |
|---------|---------------|
| **Query Q** | Ce que ce token cherche |
| **Key K** | Ce que ce token contient |
| **Value V** | Ce qu'il transmet si pertinent |

```
Attention(Q, K, V) = softmax( Q * K^T / sqrt(d_k) ) * V
```

- `Q * K^T` : score de compatibilité
- `sqrt(d_k)` : facteur d'échelle (stabilité des gradients)
- `softmax` : poids sommant à 1

### Multi-Head Attention

L'attention est exécutée **h fois en parallèle** avec des projections différentes.
Chaque tête se spécialise (syntaxe, sémantique, coréférences…).

## Transformer

Architecture ("Attention is All You Need", 2017) sans récurrence.

```
Input tokens
    |
Embedding + Positional Encoding   <- injecte l'ordre via sin/cos
    |
+---------------------------+ x N
| Multi-Head Self-Attention |   <- chaque token regarde tous les autres
| Add & LayerNorm           |   <- connexion residuelle
| Feed-Forward (Dense)      |   <- transformation non-lineaire
| Add & LayerNorm           |
+---------------------------+
    |
 Sortie
```

### Modèles fondés sur les Transformers

| Modèle | Type | Usage |
|--------|------|-------|
| **BERT** | Encodeur | Classification, NER, Q&A |
| **GPT-4** | Décodeur | Génération de texte |
| **T5** | Encodeur-Décodeur | Traduction, résumé |
| **ViT** | Encodeur (patches) | Classification d'images |
| **CLIP** | Dual-encoder | Image-texte multi-modal |

### Vision Transformer (ViT)

```
Image (224x224) -> 196 patches (14x14) -> Embeddings -> Transformer Encoder -> Classe
```

Rivalise avec les CNN sur de grands datasets.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# ============================================================
# TRANSFORMER — Composants implémentes en Keras
# ============================================================

class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.Wq = layers.Dense(embed_dim, use_bias=False)
        self.Wk = layers.Dense(embed_dim, use_bias=False)
        self.Wv = layers.Dense(embed_dim, use_bias=False)
        self.Wo = layers.Dense(embed_dim)

    def split_heads(self, x, B):
        x = tf.reshape(x, (B, -1, self.num_heads, self.head_dim))
        return tf.transpose(x, [0, 2, 1, 3])

    def call(self, x):
        B = tf.shape(x)[0]
        Q = self.split_heads(self.Wq(x), B)
        K = self.split_heads(self.Wk(x), B)
        V = self.split_heads(self.Wv(x), B)
        dk = tf.cast(self.head_dim, tf.float32)
        scores  = tf.matmul(Q, K, transpose_b=True) / tf.math.sqrt(dk)
        weights = tf.nn.softmax(scores, axis=-1)
        out = tf.matmul(weights, V)
        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, (B, -1, self.num_heads * self.head_dim))
        return self.Wo(out)

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop  = layers.Dropout(dropout)

    def call(self, x, training=False):
        x = self.norm1(x + self.drop(self.attn(x), training=training))
        x = self.norm2(x + self.drop(self.ffn(x),  training=training))
        return x

# ---- Modèle de classification ----
SEQ_LEN, EMBED_DIM, NUM_HEADS, FF_DIM, N_CLASSES = 50, 64, 4, 128, 5

inputs = tf.keras.Input(shape=(SEQ_LEN, EMBED_DIM))
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(inputs)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = layers.GlobalAveragePooling1D()(x)   # agrege la sequence
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
out = layers.Dense(N_CLASSES, activation='softmax')(x)

model_tr = tf.keras.Model(inputs, out, name='Transformer')
model_tr.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

x_dummy = np.random.randn(8, SEQ_LEN, EMBED_DIM).astype('float32')
print(f'Entree  : {x_dummy.shape}')
print(f'Sortie  : {model_tr.predict(x_dummy, verbose=0).shape}')
print(f'Params  : {model_tr.count_params():,}')
print()
print('Avantages Transformer vs LSTM :')
print('  Parallélisable (pas de récurrence) -> entrainement plus rapide')
print('  Attention globale (pas de biais de proximite)')
print('  Meilleur sur grands datasets')

---
# Partie 12 — Captioning d'Image

## Captioning d'Image — Génération de Description

### Définition

Le **captioning d'image** génère automatiquement une description textuelle d'une image.
Problème multi-modal : combine vision (CNN/ViT) et langage (RNN/Transformer).

### Architecture Encodeur-Décodeur

```
Image
  |
  v
CNN/ViT (Encodeur visuel)  ->  vecteur de features z
                                        |
                                        v
                            RNN/Transformer (Decodeur textuel)
                                        |
                                        v
                            Description generee mot par mot
```

| Étape | Rôle |
|-------|------|
| **Encodeur visuel** | Extrait les features de l'image (CNN ou ViT) |
| **Décodeur textuel** | Génère la description conditionnellement aux features |
| **Attention visuelle** | Le décodeur se concentre sur les régions pertinentes |

### Génération auto-régressive (mot par mot)

```
z  = CNN(image)
y1 = Decoder(z, <start>)              -> 'Un'
y2 = Decoder(z, <start>, 'Un')        -> 'chat'
y3 = Decoder(z, <start>, 'Un', 'chat')-> 'assis'
...                                   -> '.'
```

### Modèles fondateurs

| Modèle | Année | Innovation |
|--------|-------|------------|
| **Show and Tell** | 2014 Google | CNN + LSTM end-to-end, premier modèle |
| **Show, Attend and Tell** | 2015 | Attention visuelle sur les régions |
| **CLIP + GPT** | 2021 OpenAI | 400M paires image-texte, zero-shot |
| **BLIP** | 2022 Salesforce | Bootstrap de captions synthétiques |
| **BLIP-2** | 2023 | Q-Former connectant ViT à un LLM gelé |

## Métriques d'Évaluation

### BLEU (Bilingual Evaluation Understudy)

Mesure le **chevauchement de n-grammes** entre la description générée et les références humaines.

```
Reference : 'Un chat assis sur un canape rouge'
Generee   : 'Un chat sur un canape'

BLEU-1 (unigrammes) : 5/5 = 1.00
BLEU-2 (bigrammes)  : 3/4 = 0.75
BLEU-4              : standard en captioning (MS-COCO)
```

### CIDEr

TF-IDF sur les n-grammes → donne plus de poids aux mots rares et informatifs.
Standard dans les compétitions de captioning.

### METEOR

Prend en compte les synonymes et la racinisation.
Corrèle mieux avec le jugement humain que BLEU.

### Limites des métriques automatiques

Aucune métrique ne capture la **fluidité**, la **pertinence** ni la **créativité**.
Une évaluation humaine reste indispensable pour les publications.

## CLIP et Vision-Language Models Modernes

### CLIP (OpenAI, 2021)

Entraîné sur **400M paires (image, texte)** par apprentissage contrastif :

```
'Un chat assis' -> Text Encoder -> vecteur texte  ]
Image du chat   -> Image Encoder -> vecteur image ] maximise cosine similarity
```

**Applications :** zero-shot classification, retrieval image↔texte,
génération guidée (DALL-E, Stable Diffusion).

### BLIP-2 (2023)

Architecture légère **Q-Former** qui connecte un encodeur visuel gelé
à un LLM gelé (FlanT5). Seul le Q-Former est entraîné → très efficace.

### Comparaison des approches

| Approche | Modèle | Objectif |
|----------|--------|----------|
| **Contrastive** | CLIP | Aligner image et texte dans un espace partagé |
| **Generative** | BLIP, GPT-4V | Générer une description de l'image |
| **Hybride** | BLIP-2 | Compréhension + génération |

| VLM | Capacités |
|-----|----------|
| **GPT-4V** | Description, Q&A, OCR, analyse de documents |
| **Gemini** | Multi-modal natif (image, audio, vidéo, texte) |
| **LLaVA** | Open-source : ViT + Vicuna/Llama |
| **InstructBLIP** | BLIP-2 + instruction tuning |

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# ============================================================
# CAPTIONING — CNN Encodeur + LSTM Decodeur (Show and Tell)
# ============================================================
VOCAB_SIZE = 5000    # taille du vocabulaire
EMBED_DIM  = 256     # dimension embedding de mots
UNITS      = 512     # dimension etat LSTM
IMG_FEAT   = 2048    # features CNN (ex: InceptionV3 GlobalAvgPool)
MAX_LEN    = 30      # longueur max de la description

# ---- Encodeur visuel (CNN pre-entraine) ----
base_cnn = tf.keras.applications.InceptionV3(
    include_top=False,
    weights=None,    # mettre 'imagenet' en pratique
    pooling='avg',   # GlobalAveragePooling -> (batch, 2048)
)
base_cnn.trainable = False

# ---- Decodeur LSTM ----
class CaptionDecoder(layers.Layer):
    def __init__(self, vocab_size, embed_dim, units):
        super().__init__()
        self.embed    = layers.Embedding(vocab_size, embed_dim)
        self.lstm     = layers.LSTMCell(units)
        self.fc_feat  = layers.Dense(embed_dim)   # projette les features image
        self.fc_out   = layers.Dense(vocab_size)  # predit le prochain token

    def call(self, token, img_features, states):
        # token        : (batch, 1)   <- token courant (ex: index du mot precedent)
        # img_features : (batch, IMG_FEAT)
        # states       : [h, c]       <- etat LSTM
        tok_emb  = tf.squeeze(self.embed(token), axis=1)   # (batch, embed_dim)
        feat_emb = self.fc_feat(img_features)               # (batch, embed_dim)
        lstm_in  = tf.concat([tok_emb, feat_emb], axis=-1)  # (batch, 2*embed_dim)
        out, states = self.lstm(lstm_in, states)
        logits = self.fc_out(out)                           # (batch, vocab_size)
        return logits, states

decoder = CaptionDecoder(VOCAB_SIZE, EMBED_DIM, UNITS)

# ---- Test de forme ----
B      = 4
feats  = np.random.randn(B, IMG_FEAT).astype('float32')
token  = np.zeros((B, 1), dtype='int32')  # token <start>
states = [tf.zeros((B, UNITS)), tf.zeros((B, UNITS))]

logits, states = decoder(token, feats, states)
print(f'Features image : {feats.shape}')
print(f'Logits (1 step): {logits.shape}')   # (4, 5000)
print(f'Etat LSTM      : {[s.shape for s in states]}')

# ---- Generation greedy (1 image) ----
print('\n=== Generation auto-regressive (greedy) ===')
feat1  = feats[:1]
states = [tf.zeros((1, UNITS)), tf.zeros((1, UNITS))]
token  = np.array([[1]])   # index <start>
ids    = []
for _ in range(MAX_LEN):
    logits, states = decoder(token, feat1, states)
    pred = tf.argmax(logits, axis=-1).numpy()   # mot le plus probable
    if pred[0] == 2: break                       # token <end>
    ids.append(int(pred[0]))
    token = pred.reshape(1, 1)
print(f'IDs generes : {ids[:8]}... (a convertir avec le vocabulaire)')

# ---- Entrainement ----
print('\nEntrainement : loss = CrossEntropy(mots_cibles, logits_pred)')
print('Optimiser avec teacher forcing : en entree le mot correct (pas la prediction)')

---
# Récapitulatif — Cheat Sheet

## Architecture CNN typique

```
INPUT (H × W × 3)
  │
  ├─ [Conv2D(32) → BN → ReLU] × 2  →  MaxPool  →  Dropout(0.25)  # Bloc 1 : bords
  ├─ [Conv2D(64) → BN → ReLU] × 2  →  MaxPool  →  Dropout(0.25)  # Bloc 2 : textures
  ├─ [Conv2D(128)→ BN → ReLU] × 2  →  MaxPool  →  Dropout(0.25)  # Bloc 3 : formes
  ├─ [Conv2D(256)→ BN → ReLU]      →  MaxPool  →  Dropout(0.25)  # Bloc 4 : sémantique
  │
  ├─ GlobalAveragePooling2D()
  ├─ Dense(512) → BN → ReLU → Dropout(0.5)
  └─ Dense(N_classes, softmax)
```

## Paramètres d'entraînement typiques

| Paramètre | Valeur typique | Rôle |
|-----------|---------------|------|
| `optimizer` | Adam(lr=1e-3) | Mise à jour des poids |
| `loss` | sparse_categorical_crossentropy | Labels entiers |
| `batch_size` | 32 | Compromis vitesse/mémoire |
| `epochs` | 30–100 + EarlyStopping | Nombre max d'itérations |
| `dropout_conv` | 0.25 | Régularisation couches conv |
| `dropout_dense` | 0.5 | Régularisation couches denses |

## Diagnostic rapide

| Symptôme | Diagnostic | Solutions |
|----------|-----------|----------|
| Train acc ↑, Val acc stagne | Overfitting | Dropout ↑, Augmentation, L2 |
| Train acc et Val acc basses | Underfitting | Architecture plus grande, plus d'epochs |
| Val loss explose | LR trop grand | ReduceLROnPlateau, LR plus petit |
| Val acc oscille beaucoup | LR trop grand / batch trop petit | LR ↓, batch ↑ |
| Classe minoritaire mal classée | Déséquilibre | Class weights, Oversampling |